# 60 K spectroscopy and FN analysis (V5): AFM-anchored TVS calibration

Combines the band-edge conductance-peak spectroscopy (shared-FWHM global Gaussian fit, Method-2 errors) with the Fowler-Nordheim (FN) analysis in one notebook, and applies the AFM-anchored transition-voltage calibration.

The conductance-peak position $V_\mathrm{peak}$ is a transport feature, not a barrier height. We measure the FN transition voltage $V_T$ (minimum of $\ln(|I|/V^2)$ vs $1/V$) in the AFM endpoint, where the coherent interlayer channel is shut and the FN minimum is clean, and define the per-axis calibration $c = V_\mathrm{peak}^\mathrm{AFM}/V_T^\mathrm{AFM}$. All barrier heights are then reported as $\Phi = eV_\mathrm{peak}/c$, so $\Phi_\mathrm{AFM} = eV_T^\mathrm{AFM}$ by construction. Justification: `tmp/peer-review/TVS-barrier-calibration_thesis-section.md`.

Supersedes `Barrier_vs_canting_60K.ipynb` (which reported $\Phi = eV_\mathrm{peak}$, lever arm 1).

## 1. Imports

In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

from scripts.IV_Hscan_gaussian import load_dataframe
from scripts.barrier_canting_fit import (
    gauss_lin, weighted_mean, bin_weighted, canting_angle, sin2_half,
    TIGHT_HALF_WIDTH,
)
from scripts.global_shared_fwhm import (
    prepare_curves, fit_shared_sigma_global, add_measured_fwhm,
)
from scripts.fit_cache import load_or_compute_shared_fit

FWHM_FACTOR = 2.0 * np.sqrt(2.0 * np.log(2.0))
from scripts import fn_analysis as fn

## 2. Inputs

Saturation fields are supplied directly so the canting model can be tested cleanly. If you don't know them yet, run the V1 saturation fit (`estimate_H_sat_c` / `estimate_H_sf_b`) first and paste the numbers.

In [ ]:
TEMPERATURE = 60            # K
# TODO: set H_SAT_C and H_SF_B per temperature (placeholder values copied
# from V2; if unknown, run the V1 saturation fit
# (`estimate_H_sat_c` / `estimate_H_sf_b`) and paste the numbers here.
H_SAT_C     = 2.00          # T   c-axis saturation field
H_SF_B      = 0.30          # T   b-axis spin-flip field

# Window for the local two-step rough fit (used to find V_peak_rough
# before the global shared-sigma fit). The tight half-width sets the
# fit window around V_peak_rough that is passed to the global fit.
TIGHT_HW    = TIGHT_HALF_WIDTH      # V, see scripts.barrier_canting_fit
SIGMA_INIT  = 0.08                  # V, starting shared sigma

# Endpoint windows for AFM / FM averaging:
H_AFM_C_MAX = 0.10                                          # T
H_FM_C_MIN  = max(H_SAT_C - 0.10, 0.5 * H_SAT_C)            # T
H_AFM_B_MAX = 0.85 * H_SF_B                                  # T
H_FM_B_MIN  = 1.25 * H_SF_B                                  # T

OUT_C = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting' / 'c_scans'
OUT_B = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting' / 'b_scans'
OUT_C.mkdir(parents=True, exist_ok=True)
OUT_B.mkdir(parents=True, exist_ok=True)

print(f'TEMPERATURE = {TEMPERATURE} K')
print(f'H_SAT_C = {H_SAT_C:.3f} T, H_SF_B = {H_SF_B:.3f} T')
print(f'AFM window |H_z| < {H_AFM_C_MAX:.2f} T, FM window |H_z| > {H_FM_C_MIN:.2f} T')

## 3. Load data

In [ ]:
df_c_path = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'dataframes' / 'c_scans' / f'IV_gaussian_{TEMPERATURE}K.pkl'
df_b_path = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'dataframes' / 'b_scans' / f'IV_gaussian_{TEMPERATURE}K.pkl'
df_c = load_dataframe(df_c_path)
df_b = load_dataframe(df_b_path)

print(f'c-axis: {len(df_c)} rows, |H| up to {df_c["H"].abs().max():.2f} T')
print(f'b-axis: {len(df_b)} rows, |H| up to {df_b["H"].abs().max():.2f} T')

## 4. Global shared-FWHM fit

Step A (per curve): two-step rough fit gives $V_{\mathrm{peak,rough}}$, so we can define a tight window around the peak without contamination from the dip near 0.3 V.

Step B (joint): minimise stacked residuals across all curves with one shared $\sigma$ and per-curve $(A, V_0, m, c)$. This breaks the $\sigma$/baseline-slope degeneracy that was injecting noise into the per-curve V1 FWHM.

In [ ]:
CACHE_DIR = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting' / 'fit_cache'

curves_c, fit_c = load_or_compute_shared_fit(
    df_c, axis='c', temperature=TEMPERATURE,
    cache_dir=CACHE_DIR, tight_half_width=TIGHT_HW, sigma_init=SIGMA_INIT,
)
curves_b, fit_b = load_or_compute_shared_fit(
    df_b, axis='b', temperature=TEMPERATURE,
    cache_dir=CACHE_DIR, tight_half_width=TIGHT_HW, sigma_init=SIGMA_INIT,
)
print(f'prepared {len(curves_c)} c-axis curves and {len(curves_b)} b-axis curves')

for axis, fit in [('c', fit_c), ('b', fit_b)]:
    fwhm = FWHM_FACTOR * fit['sigma']
    fwhm_err = FWHM_FACTOR * fit['sigma_err']
    print(f'{axis}-axis  shared sigma = ({fit["sigma"]*1000:.2f} +/- {fit["sigma_err"]*1000:.2f}) mV  ->  '
          f'FWHM = ({fwhm*1000:.2f} +/- {fwhm_err*1000:.2f}) mV   '
          f'(converged={fit["success"]}, cost={fit["cost"]:.4f})')


### 4b. Measured (fit-free) FWHM

Subtract the per-curve linear baseline returned by the global fit, find the half-max crossings on the savgol-smoothed signal, and report the absolute width. This depends on the *shape* of the peak through the data, not on a Gaussian assumption -- a useful cross-check that the Gaussian sigma is not absorbing tail asymmetry.

In [ ]:
res_c = add_measured_fwhm(None, curves_c, fit_c['df'])
res_b = add_measured_fwhm(None, curves_b, fit_b['df'])

# Convenience aliases (FWHM from fit is just the shared sigma * FWHM_FACTOR).
for res, fit in [(res_c, fit_c), (res_b, fit_b)]:
    res['fwhm_fit_V']     = FWHM_FACTOR * fit['sigma']
    res['fwhm_fit_err_V'] = FWHM_FACTOR * fit['sigma_err']
    res['Phi_eV']         = res['V_peak']
    res['Phi_err_eV']     = res['V_peak_err']

print('c-axis:')
print(res_c[['H', 'V_peak', 'V_peak_err', 'fwhm_fit_V', 'fwhm_meas_V', 'fwhm_meas_err']].describe().T)
print('\nb-axis:')
print(res_b[['H', 'V_peak', 'V_peak_err', 'fwhm_fit_V', 'fwhm_meas_V', 'fwhm_meas_err']].describe().T)

## 4e. Fowler-Nordheim transition voltage and per-axis calibration

Measure $V_T^\mathrm{AFM}$ per axis from the FN minimum and define $c = V_\mathrm{peak}^\mathrm{AFM}/V_T^\mathrm{AFM}$. Every barrier height downstream is then $\Phi = eV_\mathrm{peak}/c$; the raw bias position is kept in `V_peak_bias` for the overlay plots. If the AFM $V_T$ does not resolve (higher $T$, FN regime lost), the notebook falls back to **uncalibrated** $V_\mathrm{peak}$ and prints a warning.

In [ ]:
# AFM-anchored transition-voltage calibration (per axis, this T).
def afm_endpoint_vpeak(res, h_afm_max):
    m = res['abs_H'] < h_afm_max
    mu, err, n = weighted_mean(res.loc[m, 'V_peak'], res.loc[m, 'V_peak_err'])
    return mu, err, n


def apply_calibration(res, c):
    """Phi = V_peak / c (eV). Preserve the raw bias position in 'V_peak_bias',
    then rescale V_peak / V_peak_err / Phi_* so the whole Method-2 pipeline
    operates on calibrated barrier heights and calibrated errors."""
    res = res.copy()
    if 'V_peak_bias' not in res.columns:
        res['V_peak_bias'] = res['V_peak'].astype(float)
    res['V_peak']     = res['V_peak_bias'] / c
    res['V_peak_err'] = res['V_peak_err'].astype(float) / c
    res['Phi_eV']     = res['V_peak']
    res['Phi_err_eV'] = res['V_peak_err']
    return res


# --- FN endpoints (per axis) -> V_T^AFM, B_FN ---
V_c, IA_c, IF_c, nA_c, nF_c = fn.endpoint_curves(df_c, TEMPERATURE,
                                                 h_afm_max=H_AFM_C_MAX, fm_min=H_FM_C_MIN)
V_b, IA_b, IF_b, nA_b, nF_b = fn.endpoint_curves(df_b, TEMPERATURE,
                                                 h_afm_max=H_AFM_B_MAX, fm_min=H_FM_B_MIN)
fn_afm_c, fn_fm_c = fn.fit_fn(V_c, IA_c), fn.fit_fn(V_c, IF_c)
fn_afm_b, fn_fm_b = fn.fit_fn(V_b, IA_b), fn.fit_fn(V_b, IF_b)
V_T_afm_c = fn_afm_c['V_T'] if fn_afm_c else float('nan')
V_T_afm_b = fn_afm_b['V_T'] if fn_afm_b else float('nan')

# --- V_peak^AFM (raw) and per-axis calibration factor ---
vpk_afm_c, _, _ = afm_endpoint_vpeak(res_c, H_AFM_C_MAX)
vpk_afm_b, _, _ = afm_endpoint_vpeak(res_b, H_AFM_B_MAX)
c_c = fn.calibration_factor(vpk_afm_c, V_T_afm_c)
c_b = fn.calibration_factor(vpk_afm_b, V_T_afm_b)
CALIB_OK_C, CALIB_OK_B = bool(np.isfinite(c_c)), bool(np.isfinite(c_b))

if CALIB_OK_C:
    res_c = apply_calibration(res_c, c_c)
else:
    print('WARNING: c-axis AFM V_T did not resolve; Phi is UNCALIBRATED V_peak.')
    res_c['V_peak_bias'] = res_c['V_peak']
    res_c['Phi_eV'] = res_c['V_peak']; res_c['Phi_err_eV'] = res_c['V_peak_err']
if CALIB_OK_B:
    res_b = apply_calibration(res_b, c_b)
else:
    print('WARNING: b-axis AFM V_T did not resolve; Phi is UNCALIBRATED V_peak.')
    res_b['V_peak_bias'] = res_b['V_peak']
    res_b['Phi_eV'] = res_b['V_peak']; res_b['Phi_err_eV'] = res_b['V_peak_err']

for ax, vt, c_, vpk, BA, BF in [('c', V_T_afm_c, c_c, vpk_afm_c, fn_afm_c, fn_fm_c),
                                ('b', V_T_afm_b, c_b, vpk_afm_b, fn_afm_b, fn_fm_b)]:
    bA = BA['B_FN'] if BA else float('nan')
    bF = BF['B_FN'] if BF else float('nan')
    vt_str = f'{vt*1000:.0f} meV' if np.isfinite(vt) else 'nan (no FN minimum)'
    print(f'{ax}-axis: V_peak^AFM = {vpk*1000:.0f} mV, V_T^AFM = {vt_str}, '
          f'c = {c_:.3f},  Phi_AFM = e*V_T = {vt_str}  '
          f'(B_FN^AFM={bA:.3f} V, B_FN^FM={bF:.3f} V)')


## 4g. Field-resolved FN plot (c-axis canting trajectory)

$\ln(|I|/V^2)$ vs $1/V$ for $|H_\mathrm{z}|$ bins from AFM ($H\approx 0$) through the canted states to saturated FM. This is the field-axis counterpart of the FN_five_points figure: the FN trace rotates and translates smoothly between the AFM and FM endpoints as the canting angle grows.

In [ ]:
H_BIN_CENTERS = [0.00, 0.30, 0.60, 0.90, 1.20, 1.50, 1.80, round(H_SAT_C, 2)]
fn_bins = fn.field_binned_fn(df_c, H_BIN_CENTERS, bin_hw=0.10)

cnorm = plt.Normalize(vmin=min(H_BIN_CENTERS), vmax=max(H_BIN_CENTERS))
cmap = plt.cm.coolwarm   # blue (AFM, low H) -> red (FM, high H)
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
for rec in fn_bins:
    if rec['x'] is None:
        continue
    col = cmap(cnorm(rec['H_center']))
    ax.plot(rec['x'], rec['y'], '-', color=col, lw=1.5)
    fr = rec['fit']
    if fr is not None and (rec['H_center'] == 0.0 or rec['H_center'] >= H_FM_C_MIN):
        i0, i1 = fr['i_window']
        xs = fr['x'][i0:i1]
        ax.plot(xs, fr['slope'] * xs + fr['intercept'], ls=':', color=col, lw=2.0)
if np.isfinite(V_T_afm_c):
    ax.axvline(1.0 / V_T_afm_c, color='0.3', ls='--', lw=1.0, alpha=0.7)
sm = plt.cm.ScalarMappable(norm=cnorm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(r'$|H_\mathrm{z}|$ (T)')
ax.set_xlabel(r'$1/V$ (V$^{-1}$)')
ax.set_ylabel(r'$\ln(|I|/V^2)$')
fig.tight_layout()
fig.savefig(OUT_C / f'FN_field_resolved_{TEMPERATURE}K_c.png', dpi=300)
plt.show()

print(f"{'H_center (T)':>13}  {'N':>5}  {'B_FN (V)':>10}  {'V_T (meV)':>10}")
for rec in fn_bins:
    B, VT = rec['B_FN'], rec['V_T']
    Bs = f'{B:10.3f}' if np.isfinite(B) else f"{'nan':>10}"
    VTs = f'{VT*1000:10.0f}' if np.isfinite(VT) else f"{'nan':>10}"
    print(f"{rec['H_center']:13.2f}  {rec['n_curves']:5d}  {Bs}  {VTs}")


### 4c. Setup for Method 2: FM-saturation floor and stat-error column

Method 2 (section 7) needs three inputs from `res_c` / `res_b`:

- a per-row statistical error `Phi_err_stat_eV` = the linearised covariance error already returned by the shared-FWHM global fit (`V_peak_err`),
- a pooled FM-saturation noise floor `sigma_floor` (std of $V_{\rm peak}$ across the FM branch, where $\Phi$ should be flat), used as a singleton-canting-bin fallback,
- effective FM-window minima `H_FM_*_MIN_EFF` clamped to the actual data range.

No max-inflation is applied at the per-curve level; all error inflation happens at the bin level in section 7 via the $\chi^2$ expansion helpers defined in 4d.

In [ ]:
# --- Minimal setup for Method 2: stat-error column, FM-saturation floor,
# effective FM-window minima. No per-row max-inflation (that lives at the
# bin level in section 7).

H_FM_C_MIN_EFF = min(H_FM_C_MIN, df_c['H'].abs().max() - 0.05)
H_FM_B_MIN_EFF = min(H_FM_B_MIN, df_b['H'].abs().max() - 0.05)

ROUND_H_DECIMALS = 2  # 0.01 T groups; matches bin_signed_h in section 7.


def _setup_method2_columns(res, h_fm_min):
    """Add Phi_err_stat_eV (= V_peak_err) and compute sigma_floor from the
    FM-saturation branch. Returns (res, sigma_floor, n_fm)."""
    res = res.copy()
    res['Phi_err_stat_eV'] = res['V_peak_err'].astype(float)
    fm_mask = res['abs_H'] > h_fm_min
    n_fm = int(fm_mask.sum())
    sigma_floor = (float(res.loc[fm_mask, 'V_peak'].std(ddof=1))
                   if n_fm >= 2 else float('nan'))
    return res, sigma_floor, n_fm


res_c, sigma_floor_c, n_fm_c = _setup_method2_columns(res_c, H_FM_C_MIN_EFF)
res_b, sigma_floor_b, n_fm_b = _setup_method2_columns(res_b, H_FM_B_MIN_EFF)

print(f'c-axis: FM-saturation floor = {1000*sigma_floor_c:.2f} mV (N_FM={n_fm_c})')
print(f'b-axis: FM-saturation floor = {1000*sigma_floor_b:.2f} mV (N_FM={n_fm_b})')


### 4d. Shared helpers: $\chi^{2}$ expansion factor and per-region floors

Defines `chi2_expand(N)` — the multiplicative factor that converts a sample std from $N$ samples into a one-sided 95% upper confidence limit on the true $\sigma$ — and computes the AFM / FM saturation floors with their $\chi^{2}$-expanded versions. Used by both the binned $\Phi(H_\mathrm{z})$ plot in §7b and the $\Phi$ vs $\sin^{2}(\theta/2)$ Method 2 plot in §8b.

In [ ]:
from scipy.stats import chi2 as _chi2

CL = 0.95  # one-sided upper CL on sigma


def chi2_expand(N, cl=CL):
    """Multiplicative factor: one-sided upper-(1-cl) CL on the true sigma
    given sample std s computed from N samples is s * sqrt((N-1)/chi2.ppf(1-cl, N-1)).
    Returns NaN for N < 2 (std undefined)."""
    if N < 2 or not np.isfinite(N):
        return np.nan
    return float(np.sqrt((N - 1) / _chi2.ppf(1.0 - cl, N - 1)))


def region_floor_flat(sub):
    """Std of V_peak across a region where Phi should be physically flat.
    Returns (sigma, N)."""
    if len(sub) < 2:
        return np.nan, len(sub)
    v = sub["V_peak"].values
    return float(np.std(v, ddof=1)), len(sub)


# Per-region floors for the c-axis, used by both §7b and §8b.
afm_mask_c = res_c["abs_H"] < H_AFM_C_MAX
fm_mask_c  = res_c["abs_H"] > H_FM_C_MIN_EFF

sigma_afm_c, N_afm_c = region_floor_flat(res_c[afm_mask_c])
sigma_fm_c,  N_fm_c2 = region_floor_flat(res_c[fm_mask_c])
sigma_afm_up_c = sigma_afm_c * chi2_expand(N_afm_c)
sigma_fm_up_c  = sigma_fm_c  * chi2_expand(N_fm_c2)
sigma_sat_fallback = max(sigma_afm_up_c, sigma_fm_up_c)  # canting N=1 surrogate

print(f"AFM floor: {1000*sigma_afm_c:5.2f} mV  (N={N_afm_c}, chi2-expand {chi2_expand(N_afm_c):.2f}x -> {1000*sigma_afm_up_c:5.2f} mV)")
print(f"FM  floor: {1000*sigma_fm_c:5.2f} mV  (N={N_fm_c2}, chi2-expand {chi2_expand(N_fm_c2):.2f}x -> {1000*sigma_fm_up_c:5.2f} mV)")
print(f"Canting singleton fallback (N_bin=1): max(AFM_exp, FM_exp) = {1000*sigma_sat_fallback:.2f} mV")


## 5. Spot-check the fits

Set `SPOTCHECK_FIELDS_C` / `SPOTCHECK_FIELDS_B` to the target fields you want to inspect. Each panel shows the data points in the tight window, the global-sigma Gaussian+linear fit, the linear baseline, and the measured half-max crossings.

In [ ]:
SPOTCHECK_FIELDS_C = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]
SPOTCHECK_FIELDS_B = [-0.9, -0.5, -0.15, 0.0, 0.15, 0.5, 0.9]

def spotcheck_panel(curves, res_df, sigma, fields, axis_label, save_path=None):
    n = len(fields)
    ncol = min(4, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4*ncol, 2.8*nrow), dpi=150,
                              sharey=True, squeeze=False)
    by_idx = {c['df_index']: c for c in curves}
    H_arr = res_df['H'].values
    for ax, Ht in zip(axes.ravel(), fields):
        i = int(np.argmin(np.abs(H_arr - Ht)))
        row = res_df.iloc[i]
        c = by_idx[row['df_index']]
        ax.plot(c['V'], c['N'], 'o', ms=2.5, color='0.55', alpha=0.7, label='data')
        Vg = np.linspace(c['V'].min(), c['V'].max(), 400)
        # gauss_lin's V0 argument is the Gaussian centre (V0_gauss), NOT
        # the full-model peak that we report as V_peak.
        ax.plot(Vg, gauss_lin(Vg, row['A'], row['V0_gauss'], sigma,
                              row['slope'], row['intercept']),
                '-', color=OKABE_ITO_CYCLE[2], lw=1.4, label='shared-sigma fit')
        ax.plot(Vg, row['slope'] * Vg + row['intercept'], ':', color=OKABE_ITO_CYCLE[5], lw=1.0, label='baseline')
        ax.axvline(row['V_peak_bias'], color=OKABE_ITO_CYCLE[2], ls='--', lw=0.9, alpha=0.8)
        if np.isfinite(row.get('V_left_meas', np.nan)) and np.isfinite(row.get('V_right_meas', np.nan)):
            y_half = 0.5 * row['height_meas'] + row['slope'] * 0.5 * (row['V_left_meas'] + row['V_right_meas']) + row['intercept']
            ax.hlines(y_half, row['V_left_meas'], row['V_right_meas'],
                      color='k', lw=1.4, alpha=0.7, label='FWHM (meas)')
        ax.set_title(fr'${axis_label}={row["H"]:+.2f}$ T', fontsize=9)
        ax.set_xlabel(r'$V_\mathrm{bias}$ (V)')
    for ax in axes.ravel()[n:]:
        ax.axis('off')
    axes[0, 0].set_ylabel(r'$(dI/dV)/(I/V)$')
    axes[0, 0].legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200)
    plt.show()

spotcheck_panel(curves_c, res_c, fit_c['sigma'], SPOTCHECK_FIELDS_C, 'H_Z',
                save_path=OUT_C / f'spotcheck_{TEMPERATURE}K_c.png')
spotcheck_panel(curves_b, res_b, fit_b['sigma'], SPOTCHECK_FIELDS_B, 'H_y',
                save_path=OUT_B / f'spotcheck_{TEMPERATURE}K_b.png')

## 6. Manual overrides (optional)

If a particular fit is bad (peak picked the wrong feature, baseline ran off, etc.), pin per-curve parameters here and refit. The override key is the original dataframe row index; values can include any subset of `{A, V0, m, c}`. The shared sigma is re-optimised against the remaining free parameters. Leave both dicts empty to skip the override step.

Looking up indices: `df_c.iloc[(df_c['H'] - 0.5).abs().idxmin()]` etc.

In [ ]:
OVERRIDES_C = {
    # df_c row index: dict of pinned values
    # 42: {'V0': 0.78, 'm': 0.0},
}
OVERRIDES_B = {
    # 17: {'V0': 0.62},
}

if OVERRIDES_C:
    fit_c = fit_shared_sigma_global(curves_c, sigma_init=fit_c['sigma'], fixed=OVERRIDES_C)
    res_c = add_measured_fwhm(None, curves_c, fit_c['df'])
    res_c['fwhm_fit_V']     = FWHM_FACTOR * fit_c['sigma']
    res_c['fwhm_fit_err_V'] = FWHM_FACTOR * fit_c['sigma_err']
    res_c = apply_calibration(res_c, c_c) if CALIB_OK_C else res_c
    res_c, sigma_floor_c, n_fm_c = _setup_method2_columns(res_c, H_FM_C_MIN_EFF)
    print(f'c-axis refit with {len(OVERRIDES_C)} override(s): sigma = {fit_c["sigma"]*1000:.2f} mV')

if OVERRIDES_B:
    fit_b = fit_shared_sigma_global(curves_b, sigma_init=fit_b['sigma'], fixed=OVERRIDES_B)
    res_b = add_measured_fwhm(None, curves_b, fit_b['df'])
    res_b['fwhm_fit_V']     = FWHM_FACTOR * fit_b['sigma']
    res_b['fwhm_fit_err_V'] = FWHM_FACTOR * fit_b['sigma_err']
    res_b = apply_calibration(res_b, c_b) if CALIB_OK_B else res_b
    res_b, sigma_floor_b, n_fm_b = _setup_method2_columns(res_b, H_FM_B_MIN_EFF)
    print(f'b-axis refit with {len(OVERRIDES_B)} override(s): sigma = {fit_b["sigma"]*1000:.2f} mV')

if not (OVERRIDES_C or OVERRIDES_B):
    print('No overrides applied.')

## Paper figure: representative fit overlay

Publication-style overlay of representative $(dI/dV)/(I/V)$ curves with their Gaussian + linear-baseline fits (shared $\sigma$) and the reported $V_{\mathrm{peak}}$ for a curated set of $H$ values. Two independent figures are written to `paper_figures/` (PNG + PDF) so they can be arranged side-by-side in PowerPoint.

In [ ]:
# Publication-style fit-overlay figure. Two independent figures (c-axis
# and b-axis) are saved as PNG + PDF for assembly in PowerPoint. Per
# curve we draw: full normalised dI/dV data (circles), Gaussian + linear-
# baseline fit using the global shared sigma (solid line, drawn only
# over the tight fit window), and the reported V_peak as a dashed
# vertical line.

def plot_fit_overview(curves, res_df, sigma, fields, axis_label, save_path,
                      V_lim=None, figsize=(7.5, 5.5)):
    by_idx = {c['df_index']: c for c in curves}
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    for i, H_target in enumerate(fields):
        idx = int((res_df['H'] - H_target).abs().idxmin())
        row = res_df.iloc[idx]
        c   = by_idx[row['df_index']]
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]

        # data: full normalised dI/dV (shows the band-edge in context)
        ax.plot(c['voltage_full'], c['norm_full'], 'o', color=color,
                markersize=2.5, alpha=0.45,
                label=fr'${axis_label}={row["H"]:+.2f}$ T')

        # fit: Gaussian + linear baseline on the tight window
        Vmin, Vmax = c['V'].min(), c['V'].max()
        Vfit = np.linspace(Vmin, Vmax, 400)
        ax.plot(Vfit, gauss_lin(Vfit, row['A'], row['V0_gauss'], sigma,
                                row['slope'], row['intercept']),
                '-', color=color, lw=1.4)

        # V_peak (full-model argmax) marker
        if np.isfinite(row['V_peak']):
            ax.axvline(row['V_peak_bias'], color=color, ls='--', lw=0.9, alpha=0.85)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel(r'$(dI/dV)/(I/V)$')
    if V_lim is not None:
        ax.set_xlim(*V_lim)
    ax.legend(loc='upper left', frameon=True)
    fig.tight_layout()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    fig.savefig(save_path.with_suffix('.pdf'), bbox_inches='tight')
    plt.show()


FIG_FIELDS_C = [0.0, 0.25, 0.50, 0.75, 1.0, 1.5, 2.0]
FIG_FIELDS_B = [0.0, 0.10, 0.15, 0.25, 0.50, 0.90]
FIG_DIR = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting' / 'paper_figures'

V_max_c = float(np.max([c['voltage_full'].max() for c in curves_c]))
V_max_b = float(np.max([c['voltage_full'].max() for c in curves_b]))

plot_fit_overview(curves_c, res_c, fit_c['sigma'], FIG_FIELDS_C, 'H_Z',
                  FIG_DIR / f'fit_overview_{TEMPERATURE}K_c.png',
                  V_lim=(-0.05, V_max_c))
plot_fit_overview(curves_b, res_b, fit_b['sigma'], FIG_FIELDS_B, 'H_y',
                  FIG_DIR / f'fit_overview_{TEMPERATURE}K_b.png',
                  V_lim=(-0.05, V_max_b))

print('Saved paper figures to:', FIG_DIR)

In [ ]:
# FN calibration factor recap (used for the field-dependent Phi below).
# Phi = e * V_peak / c, with c = V_peak^AFM / V_T^AFM measured per axis in 4e.
for ax_lbl, vpk, vt, c_, ok in [('c', vpk_afm_c, V_T_afm_c, c_c, CALIB_OK_C),
                                ('b', vpk_afm_b, V_T_afm_b, c_b, CALIB_OK_B)]:
    if ok:
        print(f'{ax_lbl}-axis:  c = V_peak^AFM / V_T^AFM = '
              f'{vpk*1000:.0f} mV / {vt*1000:.0f} mV = {c_:.3f}'
              f'   ->  Phi_AFM = e*V_T^AFM = {vt*1000:.0f} meV')
    else:
        print(f'{ax_lbl}-axis:  V_T^AFM unresolved -> Phi reported UNCALIBRATED')


## 7. Canonical $\Phi(H_\mathrm{z})$ — binned, Method 2

Repeats at the same $H_\mathrm{z}$ are collapsed into a single binned point with one combined error bar. The per-bin error uses **Method 2**: per-region floor with $\chi^{2}$ 95% upper CL expansion (AFM, FM pooled); canting bins use only $\sigma_{\rm repeat}\times\chi^{2}$-expand($N_{\rm bin}$), with singleton canting bins falling back to $\max(\sigma_{\rm AFM,exp},\sigma_{\rm FM,exp})$.

These binned $(\Phi, \sigma_\Phi)$ values are the canonical $\Phi(H_\mathrm{z}, T)$ used downstream (fits below saturation in section 8, T-dependence summary).

Endpoint averages ($\Phi_{\rm AFM}$, $\Phi_{\rm FM}$, $\Delta_{\rm ex}$) are inverse-variance weighted over the binned points using the Method 2 error column.

In [ ]:
# --- Bin per-curve points by signed H_z (round to 0.01 T) so each unique
# field gives one point with a combined error bar. Method 2: per-region
# chi^2-expanded floor (AFM, FM pooled); canting bins use only
# sigma_repeat x chi2-expand(N_bin). This is the only error variant kept
# in V4.

def bin_signed_h(res, sigma_floor):
    """One row per rounded signed H. Returns Phi, sigma_stat, sigma_repeat,
    sigma_floor (broadcast), N, abs_H."""
    rows = []
    for _, sub in res.groupby(res["H"].round(2)):
        v = sub["V_peak"].values
        e = sub["Phi_err_stat_eV"].values
        pos = e > 0
        e_safe = np.where(pos, e, np.median(e[pos])) if pos.any() else np.full_like(v, 1.0)
        w = 1.0 / e_safe**2
        rows.append({
            "H":       float(sub["H"].mean()),
            "abs_H":   float(sub["abs_H"].mean()),
            "Phi_eV":  float(np.sum(w * v) / np.sum(w)),
            "Phi_err_stat_eV":   float(1.0 / np.sqrt(np.sum(w))),
            "Phi_err_repeat_eV": float(np.std(v, ddof=1)) if v.size >= 2 else np.nan,
            "Phi_err_floor_eV":  sigma_floor,
            "n":       int(v.size),
        })
    return pd.DataFrame(rows).sort_values("H").reset_index(drop=True)


agg_c_signed = bin_signed_h(res_c, sigma_floor_c)

# Method 2: per-region floor with chi^2 expansion (uses sigma_afm_up_c,
# sigma_fm_up_c, sigma_sat_fallback, chi2_expand defined in section 4d).
expand_rep_m2 = agg_c_signed["n"].map(chi2_expand).values
sigma_rep_m2  = np.where(np.isfinite(agg_c_signed["Phi_err_repeat_eV"].values),
                         agg_c_signed["Phi_err_repeat_eV"].values * expand_rep_m2, 0.0)
H_abs_signed = agg_c_signed["abs_H"].values
N_signed     = agg_c_signed["n"].values
region_floor_signed = np.zeros(len(agg_c_signed))
region_floor_signed[H_abs_signed < H_AFM_C_MAX]    = sigma_afm_up_c
region_floor_signed[H_abs_signed > H_FM_C_MIN_EFF] = sigma_fm_up_c
canting_signed = (H_abs_signed >= H_AFM_C_MAX) & (H_abs_signed <= H_FM_C_MIN_EFF)
region_floor_signed[canting_signed & (N_signed < 2)] = sigma_sat_fallback
agg_c_signed["Phi_err_m2_eV"] = np.max(np.vstack([
    agg_c_signed["Phi_err_stat_eV"].values,
    sigma_rep_m2,
    region_floor_signed,
]), axis=0)


def plot_phi_vs_hz_binned(agg, err_col, save_name, method_label):
    afm  = agg["abs_H"] < H_AFM_C_MAX
    fm   = agg["abs_H"] > H_FM_C_MIN_EFF
    excl = ~(afm | fm)

    Phi_AFM, Phi_AFM_err, n_AFM = weighted_mean(agg.loc[afm, "Phi_eV"], agg.loc[afm, err_col])
    Phi_FM,  Phi_FM_err,  n_FM  = weighted_mean(agg.loc[fm,  "Phi_eV"], agg.loc[fm,  err_col])
    Delta     = Phi_AFM - Phi_FM
    Delta_err = float(np.sqrt(Phi_AFM_err**2 + Phi_FM_err**2))

    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    ax.errorbar(agg.loc[afm, "H"], 1000 * agg.loc[afm, "Phi_eV"],
                yerr=1000 * agg.loc[afm, err_col],
                fmt="o", color=OKABE_ITO_CYCLE[1], ms=5, alpha=0.9,
                ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.9, capsize=2,
                label='AFM')
    ax.errorbar(agg.loc[fm, "H"], 1000 * agg.loc[fm, "Phi_eV"],
                yerr=1000 * agg.loc[fm, err_col],
                fmt="s", color=OKABE_ITO_CYCLE[5], ms=5, alpha=0.9,
                ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.9, capsize=2,
                label='FM')
    ax.errorbar(agg.loc[excl, "H"], 1000 * agg.loc[excl, "Phi_eV"],
                yerr=1000 * agg.loc[excl, err_col],
                fmt="x", color="0.55", ms=5, alpha=0.65,
                ecolor="0.55", elinewidth=0.7, capsize=2, label="canting (excluded)")
    for mu, sig, mask, color in [(Phi_AFM, Phi_AFM_err, afm, OKABE_ITO_CYCLE[1]),
                                 (Phi_FM,  Phi_FM_err,  fm,  OKABE_ITO_CYCLE[5])]:
        if not np.isfinite(mu):
            continue
        for branch in [agg.loc[mask & (agg["H"] < 0), "H"],
                       agg.loc[mask & (agg["H"] > 0), "H"]]:
            if branch.empty:
                continue
            ax.hlines(1000 * mu, branch.min(), branch.max(), color=color, lw=1.6)
            ax.fill_between([branch.min(), branch.max()],
                            1000 * (mu - sig), 1000 * (mu + sig),
                            color=color, alpha=0.25, lw=0)
    ax.set_xlabel(r"$H_\mathrm{z}$ (T)")
    ax.set_ylabel(r"$\Phi$ (meV)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(OUT_C / save_name, dpi=300)
    plt.show()

    print(f"[{method_label}]")
    print(f"  Phi_AFM  = ({1000*Phi_AFM:.2f} +/- {1000*Phi_AFM_err:.2f}) meV (n_bins={n_AFM})")
    print(f"  Phi_FM   = ({1000*Phi_FM:.2f} +/- {1000*Phi_FM_err:.2f}) meV (n_bins={n_FM})")
    print(f"  Delta_ex = ({1000*Delta:.2f} +/- {1000*Delta_err:.2f}) meV")
    return dict(Phi_AFM=Phi_AFM, Phi_AFM_err=Phi_AFM_err,
                Phi_FM=Phi_FM,   Phi_FM_err=Phi_FM_err,
                Delta=Delta,     Delta_err=Delta_err,
                n_AFM=n_AFM, n_FM=n_FM)


hz_m2 = plot_phi_vs_hz_binned(agg_c_signed, "Phi_err_m2_eV",
                              f"Phi_vs_Hz_{TEMPERATURE}K_binned_method2.png",
                              "Method 2: per-region + chi^2 expanded")


## 8. Fits of $\Phi(H_\mathrm{z})$ below saturation

Two fits on the Method 2 binned $\Phi(H_\mathrm{z})$ restricted to $|H_\mathrm{z}| < H_{\rm sat}^\mathrm{c}$:

- **Linear**: $\Phi(H_\mathrm{z}) = a + b\,|H_\mathrm{z}|$. Slope $b$ has units meV/T; intercept $a$ is the $H_\mathrm{z}\!\to\!0$ limit.
- **$\sin^2(\theta/2)$**: $\Phi(\theta) = a + b\,\sin^2(\theta/2)$ with $\sin^2(\theta/2) = 1-(H_\mathrm{z}/H_{\rm sat}^\mathrm{c})^2$. Intercept $a$ is $\Phi_{\rm FM}^{\rm fit}$ (at $\theta=0$, saturation), and $a+b$ is $\Phi_{\rm AFM}^{\rm fit}$ (at $\theta=\pi$, $H_\mathrm{z}=0$).

Both fits are inverse-variance weighted with the Method 2 errors and report $\chi^2_{\rm red}$; parameter covariances are Birge-rescaled when $\chi^2_{\rm red}>1$.

In [ ]:
# --- Below-saturation slice; both fits + standalone cos(theta/2) publication plot. ---
def weighted_linear_fit(x, y, yerr):
    """Inverse-variance weighted fit y = a + b*x. Returns dict with a, b,
    a_err, b_err, cov_ab, chi2, chi2_red, dof, n, birge_scale."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    yerr = np.asarray(yerr, dtype=float)
    pos = yerr > 0
    safe = np.where(pos, yerr, np.median(yerr[pos]) if pos.any() else 1.0)
    w = 1.0 / safe**2
    S, Sx, Sxx = w.sum(), (w*x).sum(), (w*x*x).sum()
    Sy, Sxy = (w*y).sum(), (w*x*y).sum()
    det = S*Sxx - Sx*Sx
    a = (Sxx*Sy - Sx*Sxy) / det
    b = (S*Sxy - Sx*Sy) / det
    var_a, var_b, cov_ab = Sxx/det, S/det, -Sx/det
    resid = y - (a + b*x)
    chi2 = float(np.sum(w * resid**2))
    dof = max(int(x.size - 2), 1)
    chi2_red = chi2 / dof
    scale = max(1.0, chi2_red)
    var_a, var_b, cov_ab = var_a*scale, var_b*scale, cov_ab*scale
    return dict(a=float(a), b=float(b),
                a_err=float(np.sqrt(var_a)), b_err=float(np.sqrt(var_b)),
                cov_ab=float(cov_ab), chi2=chi2, chi2_red=float(chi2_red),
                dof=dof, n=int(x.size), birge_scale=float(scale))


below_sat = (agg_c_signed['abs_H'].values < H_SAT_C) & np.isfinite(agg_c_signed['Phi_err_m2_eV'].values)
sub = agg_c_signed.loc[below_sat].copy()
sub['cos_half'] = sub['abs_H'].values / H_SAT_C
sub['sin2_half'] = 1.0 - (sub['abs_H'].values / H_SAT_C)**2

x_cos = sub['cos_half'].values
x_sin2 = sub['sin2_half'].values
y = sub['Phi_eV'].values
yerr = sub['Phi_err_m2_eV'].values

fit_lin = weighted_linear_fit(x_cos, y, yerr)
fit_sin2 = weighted_linear_fit(x_sin2, y, yerr)

# Standalone cos(theta/2) fit (publication figure: no title).
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
xx = np.linspace(0.0, 1.0, 200)
yy = fit_lin['a'] + fit_lin['b'] * xx
band = np.sqrt(fit_lin['a_err']**2 + 2*xx*fit_lin['cov_ab'] + xx**2 * fit_lin['b_err']**2)
ax.errorbar(x_cos, 1000*y, yerr=1000*yerr, fmt='o', color=OKABE_ITO_CYCLE[2],
            ms=4, alpha=0.9, ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8,
            capsize=2, label='Data')
ax.plot(xx, 1000*yy, '-', color='black', lw=1.3, label='Linear fit')
ax.fill_between(xx, 1000*(yy-band), 1000*(yy+band), color='black', alpha=0.12, lw=0)
ax.set_xlabel(r'$\cos(\theta/2) = |H_\mathrm{z}| / H_\mathrm{sat}^\mathrm{c}$')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.set_xlim(-0.05, 1.05)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_C / f'Phi_vs_cos_half_{TEMPERATURE}K_linear_fit.png', dpi=300)
plt.show()

print(f'[linear  Phi = a + b*cos(theta/2), |H_z| < {H_SAT_C} T]')
print(f'  a       = ({1000*fit_lin["a"]:.2f} +/- {1000*fit_lin["a_err"]:.2f}) meV   [Phi at cos(theta/2)=0]')
print(f'  b       = ({1000*fit_lin["b"]:.2f} +/- {1000*fit_lin["b_err"]:.2f}) meV   [slope per unit cos(theta/2)]')
print(f'  chi^2   = {fit_lin["chi2"]:.2f}  chi^2_red = {fit_lin["chi2_red"]:.2f}  dof = {fit_lin["dof"]}  n = {fit_lin["n"]}')


### Fit-model comparison (supplementary)

The same below-saturation $\Phi(H_\mathrm{z})$ fitted with $\cos(\theta/2)$ (left) and $\sin^2(\theta/2)$ (right), with $\chi^2$ in each panel title. The linear $\cos(\theta/2)$ law is the better description of the canting dependence.

In [ ]:
# --- Side-by-side fit comparison (supplementary): chi^2 in each panel title. ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300, sharey=True)
for ax, xv, fit, xlabel, fitlabel in [
        (axes[0], x_cos, fit_lin,
         r'$\cos(\theta/2) = |H_\mathrm{z}| / H_\mathrm{sat}^\mathrm{c}$', 'Linear fit'),
        (axes[1], x_sin2, fit_sin2,
         r'$\sin^2(\theta/2) = 1 - (H_\mathrm{z}/H_\mathrm{sat}^\mathrm{c})^2$',
         r'$\sin^2(\theta/2)$ fit')]:
    xx = np.linspace(0.0, 1.0, 200)
    yy = fit['a'] + fit['b'] * xx
    band = np.sqrt(fit['a_err']**2 + 2*xx*fit['cov_ab'] + xx**2 * fit['b_err']**2)
    ax.errorbar(xv, 1000*y, yerr=1000*yerr, fmt='o', color=OKABE_ITO_CYCLE[2],
                ms=4, alpha=0.9, ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8,
                capsize=2, label='Data')
    ax.plot(xx, 1000*yy, '-', color='black', lw=1.3, label=fitlabel)
    ax.fill_between(xx, 1000*(yy-band), 1000*(yy+band), color='black', alpha=0.12, lw=0)
    ax.set_xlabel(xlabel)
    ax.set_xlim(-0.05, 1.05)
    ax.set_title(rf'$\chi^2 = {fit["chi2"]:.1f}$,  $\chi^2_\mathrm{{red}} = {fit["chi2_red"]:.2f}$')
    ax.legend(loc='best')
axes[0].set_ylabel(r'$\Phi$ (meV)')
fig.tight_layout()
fig.savefig(OUT_C / f'Phi_fit_comparison_{TEMPERATURE}K.png', dpi=300)
plt.show()


In [ ]:
# --- sin^2(theta/2) endpoint values from fit_sin2 (for the summary CSV). ---
Phi_FM_sin2 = fit_sin2['a']
Phi_FM_sin2_err = fit_sin2['a_err']
Phi_AFM_sin2 = fit_sin2['a'] + fit_sin2['b']
Phi_AFM_sin2_err = float(np.sqrt(fit_sin2['a_err']**2 + fit_sin2['b_err']**2 + 2*fit_sin2['cov_ab']))
Delta_sin2 = fit_sin2['b']
Delta_sin2_err = fit_sin2['b_err']

print('[sin^2(theta/2)  Phi = a + b*sin^2(theta/2)]')
print(f'  Phi_FM  = ({1000*Phi_FM_sin2:.2f} +/- {1000*Phi_FM_sin2_err:.2f}) meV')
print(f'  Phi_AFM = ({1000*Phi_AFM_sin2:.2f} +/- {1000*Phi_AFM_sin2_err:.2f}) meV')
print(f'  Delta   = ({1000*Delta_sin2:.2f} +/- {1000*Delta_sin2_err:.2f}) meV')
print(f'  chi^2   = {fit_sin2["chi2"]:.2f}  chi^2_red = {fit_sin2["chi2_red"]:.2f}  dof = {fit_sin2["dof"]}  n = {fit_sin2["n"]}')


## 9. $b$-axis two-state $\Phi(H_\mathrm{y})$ (binned, Method 2)

$b$-axis spin-flip is sharp around $H_{\mathrm{sf}}^{\mathrm{b}}$; AFM at $|H_\mathrm{y}| < 0.7\,H_{\mathrm{sf}}^{\mathrm{b}}$, FM at $|H_\mathrm{y}| > 1.3\,H_{\mathrm{sf}}^{\mathrm{b}}$ (clamped to the data range). Repeats at the same $H_\mathrm{y}$ are collapsed into one binned point so this plot is directly comparable to the binned c-axis $\Phi(H_\mathrm{z})$ in section 7b. Method 2 is the only error variant kept in V4. After the binned plot, per-curve weighted_mean values are also computed on `res_b` to provide the inflated b-axis $\Phi_{\mathrm{AFM,b}}$, $\Phi_{\mathrm{FM,b}}$, $\Delta_{\mathrm{ex,b}}$ that the T-dependence summary uses for the gap plot.


In [ ]:
# --- b-axis: binned Phi(H_y) with Method 2 only (mirrors c-axis section 7b).
# Repeats at the same signed H_y are collapsed via bin_signed_h (defined in
# section 7b). bin_signed_h, weighted_mean, chi2_expand, region_floor_flat,
# sigma_floor_b, H_AFM_B_MAX, H_FM_B_MIN are already in scope.

H_FM_B_MIN_EFF = min(H_FM_B_MIN, df_b['H'].abs().max() - 0.05)

# Per-curve masks (used by Method 2 binned plot and by the inflated per-curve
# weighted_mean below).
afm_b = res_b['abs_H'] < H_AFM_B_MAX
fm_b  = res_b['abs_H'] > H_FM_B_MIN_EFF

# Per-region floors for the b-axis, analogous to section 4d for the c-axis.
# The spin-flip region is excluded from the fit; only AFM and FM endpoints
# provide pooled noise estimates.
sigma_afm_b_flat, N_afm_b_flat = region_floor_flat(res_b[afm_b])
sigma_fm_b_flat,  N_fm_b_flat  = region_floor_flat(res_b[fm_b])
sigma_afm_up_b   = sigma_afm_b_flat * chi2_expand(N_afm_b_flat)
sigma_fm_up_b    = sigma_fm_b_flat  * chi2_expand(N_fm_b_flat)
sigma_sat_fallback_b = max(sigma_afm_up_b, sigma_fm_up_b)

print(f'b-axis AFM floor: {1000*sigma_afm_b_flat:5.2f} mV  (N={N_afm_b_flat}, '
      f'chi2-expand {chi2_expand(N_afm_b_flat):.2f}x -> {1000*sigma_afm_up_b:5.2f} mV)')
print(f'b-axis FM  floor: {1000*sigma_fm_b_flat:5.2f} mV  (N={N_fm_b_flat}, '
      f'chi2-expand {chi2_expand(N_fm_b_flat):.2f}x -> {1000*sigma_fm_up_b:5.2f} mV)')

# Bin per-curve points by signed H_y (0.01 T).
agg_b_signed = bin_signed_h(res_b, sigma_floor_b)

# Method 2: per-region floor with chi^2 expansion; spin-flip bins get 0
# unless they are singletons (then sigma_sat_fallback_b).
expand_rep_m2_b = agg_b_signed['n'].map(chi2_expand).values
sigma_rep_m2_b  = np.where(np.isfinite(agg_b_signed['Phi_err_repeat_eV'].values),
                           agg_b_signed['Phi_err_repeat_eV'].values * expand_rep_m2_b, 0.0)
H_abs_signed_b = agg_b_signed['abs_H'].values
N_signed_b     = agg_b_signed['n'].values
region_floor_signed_b = np.zeros(len(agg_b_signed))
region_floor_signed_b[H_abs_signed_b < H_AFM_B_MAX]    = sigma_afm_up_b
region_floor_signed_b[H_abs_signed_b > H_FM_B_MIN_EFF] = sigma_fm_up_b
spinflip_signed = (H_abs_signed_b >= H_AFM_B_MAX) & (H_abs_signed_b <= H_FM_B_MIN_EFF)
region_floor_signed_b[spinflip_signed & (N_signed_b < 2)] = sigma_sat_fallback_b
agg_b_signed['Phi_err_m2_eV'] = np.max(np.vstack([
    agg_b_signed['Phi_err_stat_eV'].values,
    sigma_rep_m2_b,
    region_floor_signed_b,
]), axis=0)


def plot_phi_vs_hy_binned(agg, err_col, save_name, method_label):
    afm  = agg['abs_H'] < H_AFM_B_MAX
    fm   = agg['abs_H'] > H_FM_B_MIN_EFF
    excl = ~(afm | fm)

    Phi_AFM, Phi_AFM_err, n_AFM = weighted_mean(agg.loc[afm, 'Phi_eV'], agg.loc[afm, err_col])
    Phi_FM,  Phi_FM_err,  n_FM  = weighted_mean(agg.loc[fm,  'Phi_eV'], agg.loc[fm,  err_col])
    Delta     = Phi_AFM - Phi_FM
    Delta_err = float(np.sqrt(Phi_AFM_err**2 + Phi_FM_err**2))

    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    ax.errorbar(agg.loc[afm, 'H'], 1000 * agg.loc[afm, 'Phi_eV'],
                yerr=1000 * agg.loc[afm, err_col],
                fmt='o', color=OKABE_ITO_CYCLE[1], ms=5, alpha=0.9,
                ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.9, capsize=2,
                label='AFM')
    ax.errorbar(agg.loc[fm, 'H'], 1000 * agg.loc[fm, 'Phi_eV'],
                yerr=1000 * agg.loc[fm, err_col],
                fmt='s', color=OKABE_ITO_CYCLE[5], ms=5, alpha=0.9,
                ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.9, capsize=2,
                label='FM')
    ax.errorbar(agg.loc[excl, 'H'], 1000 * agg.loc[excl, 'Phi_eV'],
                yerr=1000 * agg.loc[excl, err_col],
                fmt='x', color='0.55', ms=5, alpha=0.65,
                ecolor='0.55', elinewidth=0.7, capsize=2, label='spin-flip')
    for sign in (-1, +1):
        ax.axvspan(sign * H_AFM_B_MAX, sign * H_FM_B_MIN_EFF, color='0.85', alpha=0.45, lw=0)
    for mu, sig, mask, color in [(Phi_AFM, Phi_AFM_err, afm, OKABE_ITO_CYCLE[1]),
                                 (Phi_FM,  Phi_FM_err,  fm,  OKABE_ITO_CYCLE[5])]:
        if not np.isfinite(mu):
            continue
        for branch in [agg.loc[mask & (agg['H'] < 0), 'H'],
                       agg.loc[mask & (agg['H'] > 0), 'H']]:
            if branch.empty:
                continue
            ax.hlines(1000 * mu, branch.min(), branch.max(), color=color, lw=1.6)
            ax.fill_between([branch.min(), branch.max()],
                            1000 * (mu - sig), 1000 * (mu + sig),
                            color=color, alpha=0.25, lw=0)
    ax.set_xlabel(r'$H_\mathrm{y}$ (T)')
    ax.set_ylabel(r'$\Phi$ (meV)')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(OUT_B / save_name, dpi=300)
    plt.show()

    print(f'[{method_label}]')
    print(f'  Phi_AFM  = ({1000*Phi_AFM:.2f} +/- {1000*Phi_AFM_err:.2f}) meV (n_bins={n_AFM})')
    print(f'  Phi_FM   = ({1000*Phi_FM:.2f} +/- {1000*Phi_FM_err:.2f}) meV (n_bins={n_FM})')
    print(f'  Delta_ex = ({1000*Delta:.2f} +/- {1000*Delta_err:.2f}) meV')
    return dict(Phi_AFM=Phi_AFM, Phi_AFM_err=Phi_AFM_err,
                Phi_FM=Phi_FM,   Phi_FM_err=Phi_FM_err,
                Delta=Delta,     Delta_err=Delta_err,
                n_AFM=n_AFM, n_FM=n_FM)


hy_m2 = plot_phi_vs_hy_binned(agg_b_signed, 'Phi_err_m2_eV',
                              f'Phi_vs_Hy_{TEMPERATURE}K_binned_method2.png',
                              'Method 2: per-region + chi^2 expanded')


## 10. Save deliverables

Three CSVs per axis are written:

1. **Binned $\Phi(H_\mathrm{z})$** (`Phi_vs_Hz_binned_{T}K.csv`, c-axis) and **binned $\Phi(H_\mathrm{y})$** (`Phi_vs_Hy_binned_{T}K.csv`, b-axis): one row per rounded signed $H$. Holds $H$, $|H|$, $\Phi_{\rm eV}$, the Method 2 error `Phi_err_m2_eV`, the constituent error components (`Phi_err_stat_eV`, `Phi_err_repeat_eV`, `Phi_err_floor_eV`), the bin count $n$, and a region label.
2. **Fit summary** (`fit_summary_{T}K.csv`): temperature, saturation fields, shared FWHM, FM-saturation floor, Method 2 endpoint averages ($\Phi_{\rm AFM}$, $\Phi_{\rm FM}$, $\Delta_{\rm ex}$) for both axes, plus c-axis linear-fit and $\sin^2(\theta/2)$-fit parameters with errors, $\chi^2$, $\chi^2_{\rm red}$, and dof.

The summary CSV is the input the T-dependence notebook reads. Per-curve outputs from V3 (raw $V_{\rm peak}$ table, per-curve weighted_mean $\Phi$) are no longer written; the binned Method 2 values are the canonical per-temperature data.

In [ ]:
# --- Binned Phi(H_z) (c-axis), Method 2 errors. ---
agg_c_out = agg_c_signed.copy()
_abs_c = agg_c_out['abs_H'].values
agg_c_out['region'] = np.where(_abs_c < H_AFM_C_MAX, 'AFM',
                       np.where(_abs_c > H_FM_C_MIN_EFF, 'FM', 'canting'))
agg_c_out['Phi_meV']             = 1000 * agg_c_out['Phi_eV']
agg_c_out['Phi_err_m2_meV']      = 1000 * agg_c_out['Phi_err_m2_eV']
agg_c_out['Phi_err_stat_meV']    = 1000 * agg_c_out['Phi_err_stat_eV']
agg_c_out['Phi_err_repeat_meV']  = 1000 * agg_c_out['Phi_err_repeat_eV']
agg_c_out['Phi_err_floor_meV']   = 1000 * agg_c_out['Phi_err_floor_eV']
agg_c_out.to_csv(OUT_C / f'Phi_vs_Hz_binned_{TEMPERATURE}K.csv', index=False)

# --- Binned Phi(H_y) (b-axis), Method 2 errors. ---
agg_b_out = agg_b_signed.copy()
_abs_b = agg_b_out['abs_H'].values
agg_b_out['region'] = np.where(_abs_b < H_AFM_B_MAX, 'AFM',
                       np.where(_abs_b > H_FM_B_MIN_EFF, 'FM', 'spinflip'))
agg_b_out['Phi_meV']             = 1000 * agg_b_out['Phi_eV']
agg_b_out['Phi_err_m2_meV']      = 1000 * agg_b_out['Phi_err_m2_eV']
agg_b_out['Phi_err_stat_meV']    = 1000 * agg_b_out['Phi_err_stat_eV']
agg_b_out['Phi_err_repeat_meV']  = 1000 * agg_b_out['Phi_err_repeat_eV']
agg_b_out['Phi_err_floor_meV']   = 1000 * agg_b_out['Phi_err_floor_eV']
agg_b_out.to_csv(OUT_B / f'Phi_vs_Hy_binned_{TEMPERATURE}K.csv', index=False)

# --- Per-temperature summary: Method 2 endpoints + c-axis below-sat fits. ---
summary_row = {
    'temperature_K':        TEMPERATURE,
    # Geometry / inputs
    'H_sat_c_T':            H_SAT_C,
    'H_sf_b_T':             H_SF_B,
    'H_AFM_c_max_T':        H_AFM_C_MAX,
    'H_FM_c_min_T':         H_FM_C_MIN_EFF,
    'H_AFM_b_max_T':        H_AFM_B_MAX,
    'H_FM_b_min_T':         H_FM_B_MIN_EFF,
    # Shared FWHM (global fit)
    'sigma_shared_c_meV':     1000 * fit_c['sigma'],
    'sigma_shared_c_err_meV': 1000 * fit_c['sigma_err'],
    'fwhm_shared_c_meV':      1000 * FWHM_FACTOR * fit_c['sigma'],
    'fwhm_shared_c_err_meV':  1000 * FWHM_FACTOR * fit_c['sigma_err'],
    'sigma_shared_b_meV':     1000 * fit_b['sigma'],
    'sigma_shared_b_err_meV': 1000 * fit_b['sigma_err'],
    'fwhm_shared_b_meV':      1000 * FWHM_FACTOR * fit_b['sigma'],
    'fwhm_shared_b_err_meV':  1000 * FWHM_FACTOR * fit_b['sigma_err'],
    # FM-saturation floors
    'sigma_floor_c_meV':    1000 * sigma_floor_c,
    'n_FM_floor_c':         n_fm_c,
    'sigma_floor_b_meV':    1000 * sigma_floor_b,
    'n_FM_floor_b':         n_fm_b,
    # Method 2 binned endpoints, c-axis
    'Phi_AFM_c_m2_meV':       1000 * hz_m2['Phi_AFM'],
    'Phi_AFM_c_m2_err_meV':   1000 * hz_m2['Phi_AFM_err'],
    'Phi_FM_c_m2_meV':        1000 * hz_m2['Phi_FM'],
    'Phi_FM_c_m2_err_meV':    1000 * hz_m2['Phi_FM_err'],
    'Delta_ex_c_m2_meV':      1000 * hz_m2['Delta'],
    'Delta_ex_c_m2_err_meV':  1000 * hz_m2['Delta_err'],
    'n_AFM_c_m2_bins':        hz_m2['n_AFM'],
    'n_FM_c_m2_bins':         hz_m2['n_FM'],
    # Method 2 binned endpoints, b-axis
    'Phi_AFM_b_m2_meV':       1000 * hy_m2['Phi_AFM'],
    'Phi_AFM_b_m2_err_meV':   1000 * hy_m2['Phi_AFM_err'],
    'Phi_FM_b_m2_meV':        1000 * hy_m2['Phi_FM'],
    'Phi_FM_b_m2_err_meV':    1000 * hy_m2['Phi_FM_err'],
    'Delta_ex_b_m2_meV':      1000 * hy_m2['Delta'],
    'Delta_ex_b_m2_err_meV':  1000 * hy_m2['Delta_err'],
    'n_AFM_b_m2_bins':        hy_m2['n_AFM'],
    'n_FM_b_m2_bins':         hy_m2['n_FM'],
    # c-axis linear fit below saturation: Phi = a + b*|H_z|
    'lin_a_meV':            1000 * fit_lin['a'],
    'lin_a_err_meV':        1000 * fit_lin['a_err'],
    'lin_b_meV_per_T':      1000 * fit_lin['b'],
    'lin_b_err_meV_per_T':  1000 * fit_lin['b_err'],
    'lin_cov_ab':           fit_lin['cov_ab'],
    'lin_chi2':             fit_lin['chi2'],
    'lin_chi2_red':         fit_lin['chi2_red'],
    'lin_dof':              fit_lin['dof'],
    'lin_n':                fit_lin['n'],
    'lin_birge_scale':      fit_lin['birge_scale'],
    # c-axis sin^2(theta/2) fit: Phi = a + b*sin^2(theta/2)
    'sin2_a_meV':           1000 * fit_sin2['a'],
    'sin2_a_err_meV':       1000 * fit_sin2['a_err'],
    'sin2_b_meV':           1000 * fit_sin2['b'],
    'sin2_b_err_meV':       1000 * fit_sin2['b_err'],
    'sin2_cov_ab':          fit_sin2['cov_ab'],
    'sin2_chi2':            fit_sin2['chi2'],
    'sin2_chi2_red':        fit_sin2['chi2_red'],
    'sin2_dof':             fit_sin2['dof'],
    'sin2_n':               fit_sin2['n'],
    'sin2_birge_scale':     fit_sin2['birge_scale'],
    'Phi_FM_sin2_meV':      1000 * Phi_FM_sin2,
    'Phi_FM_sin2_err_meV':  1000 * Phi_FM_sin2_err,
    'Phi_AFM_sin2_meV':     1000 * Phi_AFM_sin2,
    'Phi_AFM_sin2_err_meV': 1000 * Phi_AFM_sin2_err,
    'Delta_sin2_meV':       1000 * Delta_sin2,
    'Delta_sin2_err_meV':   1000 * Delta_sin2_err,
}
summary_row.update({
    'c_calib_c': c_c, 'c_calib_b': c_b,
    'V_T_afm_c_meV': 1000 * V_T_afm_c, 'V_T_afm_b_meV': 1000 * V_T_afm_b,
    'B_FN_afm_c_V': (fn_afm_c['B_FN'] if fn_afm_c else float('nan')),
    'B_FN_fm_c_V':  (fn_fm_c['B_FN']  if fn_fm_c  else float('nan')),
    'calib_ok_c': CALIB_OK_C, 'calib_ok_b': CALIB_OK_B,
})
pd.DataFrame([summary_row]).to_csv(OUT_C / f'fit_summary_{TEMPERATURE}K.csv', index=False)

print('Saved to:')
print(' ', OUT_C / f'Phi_vs_Hz_binned_{TEMPERATURE}K.csv')
print(' ', OUT_B / f'Phi_vs_Hy_binned_{TEMPERATURE}K.csv')
print(' ', OUT_C / f'fit_summary_{TEMPERATURE}K.csv')
